In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# COMMAND ----------

silver_vehicle_positions = "abfss://silver@stspmobilitydev001.dfs.core.windows.net/sptrans/vehicle_positions"
gold_city_activity = "abfss://gold@stspmobilitydev001.dfs.core.windows.net/mobility/city_activity"

# COMMAND ----------

df = spark.read.format("delta").load(silver_vehicle_positions)

# COMMAND ----------

df_clean = df.filter(
    F.col("line_code").isNotNull()
).filter(
    F.col("latitude").isNotNull()
).filter(
    F.col("longitude").isNotNull()
)

# COMMAND ----------

city_activity = (
    df_clean
    .groupBy("event_date", "event_hour")
    .agg(
        F.countDistinct("line_code").alias("active_lines"),
        F.countDistinct("vehicle_prefix").alias("active_vehicles"),
        F.count("*").alias("total_positions"),
        F.sum(
            F.when(F.col("accessible") == True, 1).otherwise(0)
        ).alias("accessible_positions")
    )
    .withColumn(
        "accessibility_pct",
        F.round(
            F.col("accessible_positions") / F.col("total_positions"),
            4
        )
    )
)

# COMMAND ----------

city_activity.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save(gold_city_activity)

# COMMAND ----------

print("Dataset gold/city_activity criado com sucesso")

spark.read.format("delta").load(gold_city_activity).show(10)